#Transformer from Scratch with Masked Attention


In [ ]:
!pip install lightning

In [31]:
import torch
import torch.nn as nn
import torch.nn.functional as F # need for softmax for attention
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader

import lightning as L # make sure to !pip install lightning

In [5]:
from pygments import token
#use dictionary to convert words to numbers then to torch.tensor

token_to_id = {'what':0, 'is':1, 'statquest':2, 'awesome':3, '<EOS>':4}
id_to_token = dict(map(reversed, token_to_id.items()))

# convert prompts and responses into a dataset ~ eg. 'What is Statquest?' -> 'Awesome!'
# OR ANTOHER PROMPT: "Statquest is what?" -> "Awesome"
"""tokens used as input during the training comes from the prompt and the output
What, is, Statquest, EOS, awesome, EOS
"""

inputs = torch.tensor([[token_to_id["what"], #first set of prompts
                       token_to_id["is"],
                       token_to_id["statquest"],
                       token_to_id["<EOS>"],
                       token_to_id["awesome"]],

                      [token_to_id["statquest"], # second set of prompts
                       token_to_id["is"],
                       token_to_id["what"],
                       token_to_id["<EOS>"],
                       token_to_id["awesome"]]])

# the next word is the label
labels = torch.tensor([[token_to_id["is"], # first set of labels
                      token_to_id["statquest"],
                      token_to_id["<EOS>"],
                      token_to_id["awesome"],
                      token_to_id["<EOS>"]],

                      [token_to_id["is"], # second set of labels
                       token_to_id["what"],
                       token_to_id["<EOS>"],
                       token_to_id["awesome"],
                      token_to_id["<EOS>"]]])


In [6]:
# pass to a tensordataset to make a dataset
dataset = TensorDataset(inputs, labels)
dataloader = DataLoader(dataset)

## Word Embedding
Uses `nn.Embedding()`

Word embeddings capture the semantics, the meaning of the word. ***DICTIONARY DEFINITION.***

The Torch Embedding Model for this example will be dimension size 4. This means, for example, there are 4 values for *each* word. These values form a single vector. These are examples of word embeddings:

**Prompt:**

'what' -> pos=1 => `[0.2, 0.08, 0.01, 0.30]`

'is' -> pos=2 => `[0.1, 0.02, 0.00. 0.80]`

'statquest' -> pos=3 => `[1.2, 1.1, 0.4, 0.3]`

'\<EOS>' -> pos=4 => `[-0.4, 0.0, 0.0. -0.01]`

**Label:**

'is' -> pos=1 => `[1.1, 0.2, 0.3, -0.01]`

'statquest' -> pos=2 => `[1.2, -0.2, -0.3, -0.05]`

'awesome' -> pos=3 => `[0.1, 1.2, 2.3, -0.01]`

'\<EOS>' -> pos=4 => `[-0.1, 0.1, 0.1. -0.01]`





## Position Encoding - Captures order of the word through sine and cosine

Alternating sine and cosine distributions to calculate values for each token and embedding value.

Captures the order of the word in the sentence. ***LIKE The page number in a book. ***

**Here, you need the formulas.**

${pos}$ - location of the token


$PE_{pos} = cos(pos/10000)$

$PE_{pos} = sin(pos/10000)$

$d_{model}$ the number of values per token


$PE_{pos} = sin(pos/10000^{2/d_{model}})$

$PE_{cos} = cos(pos/10000^{2/d_{model}})$


And so on...for as many tokens as we want.

$PE_{(pos, 2i+1)} = sin(pos/10000^{2i/d_{model}})$

$PE_{(pos, 2i+1)} = cos(pos/10000^{2i/d_{model}})$



So now, you can enter the $pos$ for the token and the $i$ into each PE (Position Embedding) equation!


What is really super confusing is that:

For each word (e.g position token),
despite having d-dimensions, like 4, a `sin` and `cos` will be calculated for each index which is divided by the d. This is what `2\d_model` means.

So, if there are `d_model = 4`, there will be 2 indices, *not* 4. This is because:

`i=0` must calculate sin and cos

then the next index, `i=1` calculates sin and cos again.

In [12]:
i = 0
d_model = 4
pos = 0

print(2*i)
print(2*i / d_model)
print(10000**((2*i)/d_model))
print(pos/10000**((2*i)/d_model))


0
0.0
1.0
0.0


In [13]:
# write a formula
import math

pos = 0
i = 0
d_model = 4 #let's assume each token has 4 word embeddings

"""
Example:
What is statquest?

what -> pos=0
is -> pos=1

"""

PE_pos_2i = math.sin(pos/10000**((2*i)/d_model))

# PE(pos, 2i+1) = PE_pos_cos
PE_pos_cos = math.cos(pos/10000**((2*i)/d_model))

print("For index {} we calculate both sin and cos".format(i))
print("sin: PE_pos_2i", PE_pos_2i)
print("cos: PE_pos_cos", PE_pos_cos)


For index 0 we calculate both sin and cos
sin: PE_pos_2i 0.0
cos: PE_pos_cos 1.0


In [14]:
def pos_encoding(pos, i, d_model):
  PE_pos_2i = math.sin(pos/10000**((2*i)/d_model))

  # PE(pos, 2i+1) = PE_pos_cos
  PE_pos_cos = math.cos(pos/10000**((2*i)/d_model))

  print("For index {} we calculate both sin and cos".format(i))
  print("sin: PE_pos_2i", PE_pos_2i)
  print("cos: PE_pos_cos", PE_pos_cos)


In [16]:
pos_encoding(0, 1, 4)

For index 1 we calculate both sin and cos
sin: PE_pos_2i 0.0
cos: PE_pos_cos 1.0


## Now with position encodings
**Prompt:**

'what' -> pos=1 => `[0.2, 0.08, 0.01, 0.30]` ===> `[0, 1, 0, 1]`

* where `0.2` = index 0 = i=0
* where `0.08` is still i=0
* where `0.01` is i=1
* where `0.30` is still i=1

'is' -> pos=2 => `[0.1, 0.02, 0.00. 0.80]`

'statquest' -> pos=3 => `[1.2, 1.1, 0.4, 0.3]`

'\<EOS>' -> pos=4 => `[-0.4, 0.0, 0.0. -0.01]`

Note: The number of unique values is always exactly half of `d_model`. `i` is really about the index of dimension **pairs**. This is why you can't have `d_model` as odd values:


Symmetry: The algorithm is specifically built to create matched sine/cosine pairs. An odd dimension would leave one floating sine wave without its corresponding cosine twin.

Multi-Head Attention: The dimension must also be evenly divisible by the number of attention heads (typically 8, 12, or 16). An odd number would break this matrix multiplication grid.


In [20]:
# pos, index, d_model
# labels
pos_encoding(1, 0, 4)

For index 0 we calculate both sin and cos
sin: PE_pos_2i 0.8414709848078965
cos: PE_pos_cos 0.5403023058681398


In [35]:
# pos, index, d_model
# labels
pos_encoding(1, 1, 4)

For index 1 we calculate both sin and cos
sin: PE_pos_2i 0.009999833334166664
cos: PE_pos_cos 0.9999500004166653


**Label:**

'is' -> pos=1 => `[1.1, 0.2, 0.3, -0.01]` ===> `[0.84, 0.54, 0.009, 0.99]'

'statquest' -> pos=2 => `[1.2, -0.2, -0.3, -0.05]`

'awesome' -> pos=3 => `[0.1, 1.2, 2.3, -0.01]`

'\<EOS>' -> pos=4 => `[-0.1, 0.1, 0.1. -0.01]`

## Position Encoding


Make a matrix of positional embeddings. It will look like this:

```
torch.tensor([[0.0000, 1.0000],
              [0.8415, 0.5403],
              [0.9001, -0.4212]])
```

The first column are the sine values. The second column are the cosine values.

In [21]:
class PositionEncoding(nn.Module):
  def __init__(self, d_model=2, max_len=6): #max_len max number of tokens
    super().__init__()

    # pe is a matrix of position encoding values
    pe = torch.zeros(max_len, d_model) # matrix of zeros size [6,2]

    position = torch.arange(start=0, end=max_len, step=1).float().unsqueeze(1) #unsqueeze turns into a column, if max_len = 3 like torch.tensor[[0.], [1.], [2.]]
    embedding_index = torch.arange(start=0, end=d_model, step=2).float() # 2i

    div_term = 1/torch.tensor(100000.0)**(embedding_index / d_model)

    pe[:, 0::2] = torch.sin(position * div_term) # 2 means every other column
    pe[:, 1::2] = torch.cos(position * div_term)

    self.register_buffer('pe', pe) # puts on gpu

  def forward(self, word_embeddings):
    # add positional embedding to word embedding
    return word_embeddings + self.pe[:word_embeddings.size(0), :]

# Masked Self Attention

In [22]:
class Attention(nn.Module):
  def __init__(self, d_model=2):
    super().__init__()

    # weight matrix for Q
    self.W_q = nn.Linear(in_features=d_model, out_features=d_model, bias=False) #in: row, out: columns
    # weight matrix for K
    self.W_k = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
    # weight matrix for V
    self.W_v = nn.Linear(in_features=d_model, out_features=d_model, bias=False)

    self.row_dim=0
    self.col_dim = 1 #track indices for rows and columns

    # forward
  def forward(self, encodings_for_q, encodings_for_k, encodings_for_v, mask=None):
    q = self.W_q(encodings_for_q)
    k = self.W_k(encodings_for_k)
    v = self.W_v(encodings_for_v)

    sims = torch.matmul(q, k.transpose(dim0 = self.row_dim, dim1=self.col_dim))
    #scale
    scaled_sims = sims/torch.tensor(k.size(self.col_dim)**0.5) # ^ 1/2 is square root

    """
    Formula: MaskedAttention(Q,K,V) = Softmax(QK^T / sqrt(d_k) + M)*V
    - where "M" is the mask


    tensor([[False, True, True],
            [False, False, True],
            [False, False, False]])

    Replace with -1e9 for negative infinity proxy
    replace False = 0

    tensor([0, -1e9, -1e9],
            [0, 0, -1e9],
            [0, 0, 0]])

    """
    #mask keeps from cheating to look ahead
    if mask is not None:
      scaled_sims = scaled_sims.masked_fill(mask=mask, value=-1e9)

    attention_percents = F.softmax(scaled_sims, dim=self.col_dim)
    attention_scores = torch.matmul(attention_percents, v)

    return attention_scores


In [40]:
class DecoderOnlyTransformer(L.LightningModule):

  def __init__(self, num_tokens=4, d_model=2, max_len=6):
    super().__init__()

    # word embedding
    self.we = nn.Embedding(num_embeddings=num_tokens, embedding_dim=d_model)
    self.pe = PositionEncoding(d_model=d_model, max_len = max_len)
    self.self_attention = Attention(d_model=d_model)
    self.fc_layer = nn.Linear(in_features=d_model, out_features=num_tokens) #full connected layer

    # woohoo loss
    self.loss = nn.CrossEntropyLoss() #softmax

  def forward(self, token_ids):
    word_embeddings = self.we(token_ids)
    position_encoded = self.pe(word_embeddings)

    # Create mask on the same device as token_ids
    mask = torch.tril(torch.ones((token_ids.size(dim=0), token_ids.size(dim=0)), device=token_ids.device)) #tri-L like lower triangle
    mask = mask == 0 #convert 1~ False and 0 ~ Trues

    self_attention_values = self.self_attention(position_encoded,
                                                position_encoded,
                                                position_encoded,
                                                mask=mask)

    residual_connection_values = position_encoded + self_attention_values
    # final layer
    fc_layer_output = self.fc_layer(residual_connection_values)

    return fc_layer_output

  def configure_optimizers(self):
    return Adam(self.parameters(), lr=0.001)

  def training_step(self, batch, batch_idx):
    input_tokens, labels=batch
    output = self(input_tokens[0]) # Use 'self' for forward pass
    loss = self.loss(output, labels[0])
    return loss

# Training

with Adam

# Model


In [43]:
model = DecoderOnlyTransformer(num_tokens=len(token_to_id), d_model=2, max_len=6)

# prompt
model_input = torch.tensor([token_to_id["what"],
                            token_to_id["is"],
                            token_to_id["statquest"],
                            token_to_id["<EOS>"]])
# how many tokens
input_length = model_input.size(dim=0) # our little model only has 6 token window

predictions = model(model_input)
predicted_id = torch.tensor([torch.argmax(predictions[-1, :])]) #argmax identifies output token with largest value, will be the first token generated
predicted_ids = predicted_id

max_length = 6
# loop keep generating output tokens
for i in range(input_length, max_length):
  if (predicted_id == token_to_id["<EOS>"]):
    break

  model_input= torch.cat((model_input, predicted_id)) # each time we generate a new output, we add to the output tokens so far (full context)

  predictions = model(model_input)
  predicted_id = torch.tensor([torch.argmax(predictions[-1,:])])
  predicted_ids = torch.cat((predicted_ids, predicted_id))

print("Predicted Tokens:\n")
for id in predicted_ids:
  print("\t", id_to_token[id.item()])

Predicted Tokens:

	 awesome
	 <EOS>


In [42]:
# Re-create dataset and dataloader to ensure correct type is passed to trainer
dataset = TensorDataset(inputs, labels)
dataloader = DataLoader(dataset)

trainer = L.Trainer(max_epochs=30)
trainer.fit(model, train_dataloaders=dataloader)

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ we             │ Embedding        │     10 │ train │     0 │
│ 1 │ pe             │ PositionEncoding │      0 │ train │     0 │
│ 2 │ self_attention │ Attention        │     12 │ train │     0 │
│ 3 │ fc_layer       │ Linear           │     15 │ train │     0 │
│ 4 │ loss           │ CrossEntropyLoss │      0 │ train │     0 │
└───┴────────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 37                                                                                               
Non-trainable params: 0                                                                                            
Total params: 37                                                                                                   
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 8                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loops/fit_loop.py:317: The number of training batches (2) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.
INFO: `Trainer.fit` stopped: `max_epochs=30` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=30` reached.


Then you rerun `model` again.